# Fig. 3 — Cumulant expansion in the $(w^{\rm p}, N_g)$ plane

Steady state of the full lasing model with a second-order cumulant expansion on a 150 × 150 grid of pump rate $w^{\rm p}$ × gain atoms $N_g$, at $N_c = 3000$ and $r = 0.5$: photon number (**panel a**), gain-atom population and collective correlations (**panel b**) and emission linewidth (**panel c**), plus the validity conditions of the adiabatic eliminations.

**Outputs** (`../data/P_Ng/`): `P.npy`, `n.npy`, `iterations.npy`, `linewidth.npy`, `SdagS.npy`, `pop_gain.npy`, `purcell.npy`, `cond_qubits.npy`, `cond_adiab_r.npy`, `cond_adiab_u.npy`.

The loop over $w^{\rm p}$ is parallelised with `Threads.@threads`: start Julia with several threads (e.g. `julia -t auto`).

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using LinearAlgebra
using ProgressMeter
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_O = 150
num_Nl = 150

# Rb87 level structure (B in G, frequencies in 2π·GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = 3000
N_atom_II = logrange(10,10000,num_Nl)
N_atom = N_atom_I .+ N_atom_II'

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.5 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

P = logrange(1e-6,1.798e-2,num_O)
P_T = P.+P_eff

C = (2*N_atom_II'.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II'
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

purc = 4*g_02./(dec_f.+kappa)

## Cumulant expansion

Squeezed cavity mode $a_s$ (with $a = a_s\cosh r - a_s^\dagger\sinh r$), auxiliary mode $b$, $N_c$ cooling atoms and $N_g$ gain atoms, after the adiabatic elimination of the excited levels. Second-order cumulant expansion with QuantumCumulants.jl.

In [ ]:
@rnumbers Δ_cav_b Δ_0_b g_b G χ_b κ κ_b R Nc Nl pc γ_ϕc γc pl γ_ϕl γl Λ # Parameters

h_cav = FockSpace(:cavity) # Hilbert space of the cavity
h_atom_C = NLevelSpace(:atom_c,(:g,:e)) # Hilbert space of the atom
h_atom_L = NLevelSpace(:atom_l,(:g,:e)) # Hilbert space of the atom
h_cav_b = FockSpace(:cavity_b)
h = h_cav⊗h_atom_C⊗h_cav_b⊗h_atom_L # Complete Hilbert space

σc(x,y,k) = IndexedOperator(Transition(h, :σc, x, y, 2), k) # Operators
σl(x,y,k) = IndexedOperator(Transition(h, :σl, x, y, 4), k) # Operators
as = Destroy(h,:as,1)
a = as*cosh(R)-as'*sinh(R)
b = Destroy(h,:b,3)

ic = Index(h, :ic, Nc, h_atom_C)
jc = Index(h, :jc, Nc, h_atom_C)
il = Index(h, :il, Nl, h_atom_L)
jl = Index(h, :jl, Nl, h_atom_L)

Hc = Δ_cav_b*b'*b + ∑(Δ_0_b*σc(:e,:g,ic)*σc(:g,:e,ic) + G*(as*σc(:e,:g,ic)+as'*σc(:g,:e,ic)) + g_b*(b*σc(:e,:g,ic)+b'*σc(:g,:e,ic)) + χ_b*σc(:e,:g,ic)*σc(:g,:e,ic)*b'*b,ic) # Hamiltonian
Hl = ∑(G*(as*σl(:e,:g,il)+as'*σl(:g,:e,il)),il)

H = Hc + Hl #+ Λ*(im*as-im*as')

D = [a;b;σc(:e,:g,ic);σc(:e,:e,ic)-σc(:g,:g,ic);σc(:g,:e,ic);σl(:e,:g,il);σl(:e,:e,il)-σl(:g,:g,il);σl(:g,:e,il)]
rates = [κ; κ_b; pc; γ_ϕc; γc; pl; γ_ϕl; γl]

In [ ]:
ops = [as] # Operators whose equations I want to derive
eqs = meanfield(ops,H,D;rates=rates,order=2) # Derive set of equations and factorize correlators for closing the set of equations
me_comp = complete!(eqs) # Automatically complete the system
eqs_sc = scale(me_comp) # Rescales the equations using symmetries 
@named sys = System(eqs_sc)

## Steady state

Each point is integrated repeatedly (up to 10 times) until $\langle a_s^\dagger a_s\rangle$ and $\langle a_s^2\rangle$ stop changing over the integration window, re-using the previous solution as initial condition; observables are then time-averaged over one more window. Points that do not converge (`iterations > 10`) are non-stationary.

In [ ]:
n = zeros(num_O,num_Nl) # Photon number of the squeezed mode a_s
M = zeros(num_O,num_Nl) # Excited-state population of one gain atom
pob_cool = zeros(num_O,num_Nl) # Excited-state population of one cooling atom
n_b = zeros(num_O,num_Nl) # Photon number of mode b
sdb = zeros(ComplexF64,num_O,num_Nl) # <σc⁺ b>
corr = zeros(num_O,num_Nl) # <S⁺S> of the gain atoms

it = zeros(num_O,num_Nl)
dist = zeros(num_O,num_Nl)
dist2 = zeros(num_O,num_Nl)

param_list = (Δ_cav_b, Δ_0_b, g_b, G, χ_b, κ, κ_b, R, Nc, Nl, pc, γ_ϕc, γc, pl, γ_ϕl, γl, Λ)

progress = Progress(num_O*num_Nl)
safe_lock = ReentrantLock()

Threads.@threads for c in 1:num_O

    u0 = zeros(ComplexF64, length(eqs_sc))
    sol = nothing
    p0 = nothing
    dict = nothing
    prob = nothing

    for l in 1:num_Nl

        dist[c,l] = 10
        dist2[c,l] = 1

        while (dist[c,l]>1 || dist2[c,l]>0.01) && it[c,l] < 10

            p0 = (w_c_b[l], w_0_b[l], g[l], sqrt(g_02[l]), chi_b[l], kappa, kappa_b, r, N_atom_I, N_atom_II[l], P_eff[l], dep_eff[l]+dep_eff_b[l], dec_eff[l]+dec_eff_b[l], P_T[c,l], dep_T[l], dec_eff[l], 0)
            dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

            prob = ODEProblem(sys,dict,(0.0,30000.0))
            sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

            lock(safe_lock) do
                dist_new = maximum(abs.(sol[as'as]))-minimum(abs.(sol[as'as]))
                dist2[c,l] = maximum(abs.(sol[as*as]))-minimum(abs.(sol[as*as]))

                if abs(dist_new - dist[c,l])/dist[c,l]<0.02 && dist[c,l] > 10
                    dist[c,l] = dist_new
                    it[c,l] = 10
                else
                    dist[c,l] = dist_new
                end
            end

            u0 = sol.u[end]

            it[c,l] = it[c,l]+1

            if sol.retcode != ReturnCode.Success
                dist[c,l] = 0.0
                break
            end

        end

        if sol.retcode != ReturnCode.Success

            n[c,l] = NaN
            M[c,l] = NaN
            pob_cool[c,l] = NaN
            n_b[c,l] = NaN
            sdb[c,l] = NaN
            corr[c,l] = NaN

        else
            prob = ODEProblem(sys,dict,(0.0,30000.0))
            sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

            u0 = sol.u[end]

            n_t = length(sol.t)

            lock(safe_lock) do
                n[c,l] = sum(real.(sol[as'*as]))/n_t
                M[c,l] = sum(real.(sol[σl(:e,:e,1)]))/n_t
                pob_cool[c,l] = sum(real.(sol[σc(:e,:e,1)]))/n_t
                n_b[c,l] = sum(real.(sol[b'b]))/n_t
                sdb[c,l] = sum(sol[σc(:e,:g,1)*b])/n_t
                corr[c,l] = sum((real.(sol[σl(:e,:g,1)*σl(:g,:e,1)]).+(N_atom_II[l]-1).*real.(sol[σl(:e,:g,1)*σl(:g,:e,2)])).*N_atom_II[l])/n_t
            end

        end

        next!(progress)
    end
end

## Validity conditions

In [ ]:
n_bare = n*cosh(2*r).+sinh(r)^2 # Photon number of mode a
cond_adiab_r = -gr*sqrt.(N_atom.*(n_bare.+1))./delta_r # << 1: adiabatic elimination of level r
cond_adiab_u = -gb*sqrt.(N_atom_I.*(n_b.+1))./delta_u # << 1: adiabatic elimination of level u

# Excited-state population of the cooling atoms
term1 = N_atom_I.^2 .*dec.^2 .- 4*w_0_b.^2 .- 16*n.*g_02
term2 = 16*N_atom_I.^2 .*dec.^2 .* w_0_b.^2
effSz = (-sqrt.((term1.+sqrt.(term2.+term1.^2))./(2*N_atom_I.^2 .* dec.^2)).+1)*0.5

## Emission linewidth

Twice the decay rate of the slowest mode of the linearised mean-field equations, evaluated with the steady-state populations.

In [ ]:
function mf_matrix(κ,κb,g_0,g_b,Δ,Nc,Nl,P_c,γ_c,ϕ_c,P_l,γ_l,ϕ_l,χ,M_c,M_l,nb,sdb)

    M = zeros(ComplexF64,4,4)
    sz_c = 2*M_c-1
    sz_l = 2*M_l-1
    Γ_l = P_l+γ_l+4*ϕ_l
    Γ_c = P_c+γ_c+4*ϕ_c

    M[1,1]=-κ/2
    M[1,3]=im*g_0*Nc
    M[1,4]=im*g_0*Nl

    M[2,2]=-κb/2+im*Δ+im*Nc*χ*M_c
    M[2,3]=im*g_b*Nc

    M[3,1]=-im*sz_c*g_0
    M[3,2]=-im*sz_c*g_b+im*χ*sdb
    M[3,3]=-(Γ_c/2-im*Δ-im*χ*nb)

    M[4,1]=-im*g_0*sz_l
    M[4,4]=-Γ_l/2

    return M
end

eigs = zeros(ComplexF64,num_O,num_Nl,4)
gap = zeros(num_O,num_Nl)

Threads.@threads for i in 1:num_O
    for j in 1:num_Nl
        if !isfinite(n[i,j]) # Failed integration
            gap[i,j] = NaN
            continue
        end
        eigs[i,j,:] = eigvals(mf_matrix(kappa,kappa_b,sqrt(g_02[j]),g[j],w_0_b[j],N_atom_I,N_atom_II[j],P_eff[j],dec_eff[j]+dec_eff_b[j],dep_eff[j]+dep_eff_b[j],P_T[i,j],dec_T[j],dep_T[j],chi_b[j],pob_cool[i,j],M[i,j],n_b[i,j],sdb[i,j]))
        gap[i,j] = -2*real.(eigs[i,j,4])
        if gap[i,j] < 0 || it[i,j]>10
            gap[i,j] = NaN
        end
    end
end

## Save

In [ ]:
data_dir = joinpath("..", "data", "P_Ng") # Relative to this notebook

npzwrite(joinpath(data_dir, "P.npy"), collect(P))
npzwrite(joinpath(data_dir, "n.npy"), n)
npzwrite(joinpath(data_dir, "iterations.npy"), it)
npzwrite(joinpath(data_dir, "linewidth.npy"), gap)
npzwrite(joinpath(data_dir, "SdagS.npy"), corr)
npzwrite(joinpath(data_dir, "pop_gain.npy"), M)
npzwrite(joinpath(data_dir, "purcell.npy"), purc)
npzwrite(joinpath(data_dir, "cond_qubits.npy"), effSz)
npzwrite(joinpath(data_dir, "cond_adiab_r.npy"), cond_adiab_r)
npzwrite(joinpath(data_dir, "cond_adiab_u.npy"), cond_adiab_u)